# 12 - Advanced Models: Markov Switching, Neural Gating, Transformer

This notebook sketches and implements advanced research extensions:

1. Markov-switching model as an alternative to HMM
2. Neural gating network for Mixture of Experts
3. Transformer encoder using 60-day rolling sequences

Input preference:

- `../data/processed/features_macro_enriched.csv` if available
- otherwise `../data/processed/regime_features_all_assets.csv`

## Important Terms

- **Markov-switching model**: statistical model where return behavior changes across hidden regimes.
- **Bayesian HMM**: HMM variant that estimates uncertainty over parameters; not implemented here unless using specialized libraries.
- **Neural gating network**: neural model that learns expert weights instead of relying only on HMM probabilities.
- **Transformer encoder**: sequence model that reads a window of past observations, such as 60 trading days.

This notebook is intentionally experimental. It is the research playground after the stable pipeline is complete.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

plt.style.use("seaborn-v0_8-whitegrid")
DATA_DIR = Path("../data/processed")

## 1. Load Data

In [2]:
macro_path = DATA_DIR / "features_macro_enriched.csv"
regime_path = DATA_DIR / "regime_features_all_assets.csv"

if macro_path.exists():
    df = pd.read_csv(macro_path, parse_dates=["Date"])
    print("Loaded macro-enriched features")
else:
    df = pd.read_csv(regime_path, parse_dates=["Date"])
    print("Loaded regime all-assets features")

df = df.sort_values(["ticker", "Date"]).reset_index(drop=True)
print(df.shape)
print(df["ticker"].value_counts())
df.head()

Loaded macro-enriched features
(3694, 30)
ticker
^GSPC    1404
^IXIC    1404
^NSEI     886
Name: count, dtype: int64


,Date,ticker,Open,High,Low,Close,Volume,return_1d,log_return_1d,volatility_7d,...,drawdown,high_low_range,volume_change,vix_close,vix_return_1d,vix_volatility_30d,target_return_1d,target_direction_1d,india_vix_close,asset_vix_close
0,2010-03-30,^GSPC,1173.750000,1177.829956,1168.920044,1173.270020,4.085000e+09,0.000043,0.000043,0.004552,...,-0.000767,0.007622,-0.066409,17.129999,-0.026151,0.033997,-0.003273,0,19.799999,17.129999
1,2010-03-31,^GSPC,1171.750000,1174.560059,1165.770020,1169.430054,4.484340e+09,-0.003273,-0.003278,0.004594,...,-0.004037,0.007540,0.097758,17.590000,0.026854,0.034446,0.007414,1,19.870001,17.590000
2,2010-04-01,^GSPC,1171.229980,1181.430054,1170.689941,1178.099976,4.006870e+09,0.007414,0.007386,0.004654,...,0.000000,0.009174,-0.106475,17.469999,-0.006822,0.033442,0.007928,1,17.620001,17.469999
3,2010-04-05,^GSPC,1178.709961,1187.729980,1178.709961,1187.439941,3.881620e+09,0.007928,0.007897,0.004543,...,0.000000,0.007652,-0.031259,17.020000,-0.025758,0.033352,0.001684,1,17.170000,17.020000
4,2010-04-06,^GSPC,1186.010010,1191.800049,1182.770020,1189.439941,4.086180e+09,0.001684,0.001683,0.004200,...,0.000000,0.007635,0.052700,16.230000,-0.046416,0.034203,-0.005877,0,17.200001,16.230000


## 2. Markov-Switching Model

This uses `statsmodels` if available.

Unlike `hmmlearn`, Markov-switching regression is a classical econometric model often used for regime-switching returns.

In [3]:
PRIMARY_TICKER = "^GSPC"
asset = df[df["ticker"] == PRIMARY_TICKER].dropna(subset=["return_1d"]).copy()
asset = asset.sort_values("Date").reset_index(drop=True)

try:
    import statsmodels.api as sm
    from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

print("statsmodels available:", HAS_STATSMODELS)
print(asset.shape)

statsmodels available: False
(1404, 30)


In [4]:
if HAS_STATSMODELS:
    y = asset["return_1d"] * 100

    ms_model = MarkovRegression(
        y,
        k_regimes=3,
        trend="c",
        switching_variance=True,
    )

    ms_result = ms_model.fit(search_reps=20, em_iter=20, disp=False)
    print(ms_result.summary())

    smoothed_probs = ms_result.smoothed_marginal_probabilities
    asset["ms_regime"] = smoothed_probs.idxmax(axis=1).values

    for col in smoothed_probs.columns:
        asset[f"ms_regime_{col}_prob"] = smoothed_probs[col].values

    asset[["Date", "return_1d", "ms_regime"]].head()
else:
    print("Install statsmodels to run this section: %pip install statsmodels")

Install statsmodels to run this section: %pip install statsmodels


## 3. Neural Gating Network

The earlier soft MoE used HMM probabilities as weights.

A neural gating network learns weights directly from market features.

This lightweight version uses logistic regression as a simple learned gate. You can later replace it with PyTorch.

In [5]:
required_cols = [
    "return_1d",
    "volatility_30d",
    "momentum_30d",
    "ma_50_ratio",
    "drawdown",
    "target_direction_1d",
]

if "asset_vix_close" in df.columns:
    vix_col = "asset_vix_close"
elif "vix_close" in df.columns:
    vix_col = "vix_close"
else:
    vix_col = None

if vix_col:
    required_cols.append(vix_col)

model_df = df.dropna(subset=required_cols).copy()
model_df["target_direction_1d"] = model_df["target_direction_1d"].astype(int)

feature_cols = [c for c in required_cols if c != "target_direction_1d"]

split_idx = int(len(model_df) * 0.8)
train = model_df.iloc[:split_idx]
test = model_df.iloc[split_idx:]

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_test = scaler.transform(test[feature_cols])
y_train = train["target_direction_1d"]
y_test = test["target_direction_1d"]

neural_gate_proxy = LogisticRegression(max_iter=1000, class_weight="balanced")
neural_gate_proxy.fit(X_train, y_train)
up_prob = neural_gate_proxy.predict_proba(X_test)[:, 1]

print("Learned gate proxy accuracy:", accuracy_score(y_test, up_prob >= 0.5))
print("Learned gate proxy F1:", f1_score(y_test, up_prob >= 0.5))
try:
    print("Learned gate proxy ROC-AUC:", roc_auc_score(y_test, up_prob))
except Exception as exc:
    print(exc)

Learned gate proxy accuracy: 0.4501510574018127
Learned gate proxy F1: 0.45180722891566266
Learned gate proxy ROC-AUC: 0.47100439882697953


## 4. Transformer Sequence Dataset

This section prepares 60-day rolling windows.

Input shape:

```text
samples x 60 days x features
```

Target:

```text
next-day direction
```

This is the correct data shape for a Transformer encoder.

In [6]:
def make_sequence_dataset(data, ticker, feature_cols, target_col, window=60):
    temp = data[data["ticker"] == ticker].dropna(subset=feature_cols + [target_col]).copy()
    temp = temp.sort_values("Date").reset_index(drop=True)

    X, y, dates = [], [], []

    values = temp[feature_cols].values
    targets = temp[target_col].values
    date_values = temp["Date"].values

    for i in range(window, len(temp)):
        X.append(values[i-window:i])
        y.append(targets[i])
        dates.append(date_values[i])

    return np.asarray(X), np.asarray(y), np.asarray(dates)

sequence_feature_cols = feature_cols
window = 60

X_seq, y_seq, seq_dates = make_sequence_dataset(
    model_df,
    ticker="^GSPC",
    feature_cols=sequence_feature_cols,
    target_col="target_direction_1d",
    window=window,
)

print("X sequence shape:", X_seq.shape)
print("y sequence shape:", y_seq.shape)
print("First sequence date:", seq_dates[0] if len(seq_dates) else None)

X sequence shape: (563, 60, 6)
y sequence shape: (563,)
First sequence date: 2011-12-01T00:00:00.000000


## 5. Optional PyTorch Transformer

This cell only runs if PyTorch is installed.

For a polished final project, this would become its own notebook with train/validation curves and proper regularization.

In [7]:
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False

print("PyTorch available:", HAS_TORCH)

if HAS_TORCH and len(X_seq) > 100:
    class TransformerDirectionModel(nn.Module):
        def __init__(self, n_features, d_model=32, nhead=4, num_layers=2):
            super().__init__()
            self.input_projection = nn.Linear(n_features, d_model)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                batch_first=True,
                dropout=0.1,
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.classifier = nn.Linear(d_model, 1)

        def forward(self, x):
            x = self.input_projection(x)
            x = self.encoder(x)
            x = x[:, -1, :]
            return self.classifier(x).squeeze(-1)

    print("TransformerDirectionModel is defined and ready for training.")
else:
    print("Install PyTorch to train transformer: pip install torch")

PyTorch available: False
Install PyTorch to train transformer: pip install torch


## Research Interpretation

This notebook gives you the path from strong classical ML into deeper research territory.

Recommended order for real implementation:

1. Markov-switching as econometric comparison.
2. Neural gating network as MoE upgrade.
3. Transformer encoder after collecting more data and adding macro features.